In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## preprocess and embedding

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [5]:
control_key = "is_control"
condition_keys = "target_gene"
condition_rep_keys = "gene_embeddings"
random_seed = 42
dataset_name = "ArcVirtualCell"
#sample_rep = "X_pca" 
#sample_rep = "X_scVI" 
sample_rep = "X_flatvi"
#sample_rep = "X_state"

In [6]:
#!state emb transform --model-folder ./data/SE-600M --input ./data/raw/adata_Training.h5ad --output ./data/raw/adata_Training_state_emb.h5ad
# in obsm['X_state']

In [7]:
if sample_rep == "X_state":
    filePath = './data/raw/adata_Training_state_emb.h5ad'
else:
    filePath = './data/raw/adata_Training.h5ad'
adata = sc.read_h5ad(filePath)
#adata = adata[adata.obs.sample(frac=0.1, random_state=42).index].to_memory()
adata.layers["counts"] = adata.X.copy()
print(adata)

AnnData object with n_obs × n_vars = 221273 × 18080
    obs: 'target_gene', 'guide_id', 'batch'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt', 'ribo', 'n_cells'
    layers: 'counts'


In [8]:
adata.obs[control_key] = (adata.obs[condition_keys] == "non-targeting")
gene_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    183097
True      38176
Name: count, dtype: int64


In [9]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.1
gene_list = list(gene_list)
zero_shot = False

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    adata_pert = adata[adata.obs[control_key] == False].copy()
    y = adata_pert.obs[condition_keys].astype(str).values
    idx = np.arange(adata_pert.n_obs)
    train_idx, test_idx = train_test_split(
        idx,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata_pert[train_idx].copy()
    adata_test = adata_pert[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    print(gene_list)
    del adata, adata_pert
else:
    # 按基因分割 zero-shot
    n_test = max(1, int(len(gene_list) * test_ratio))
    test_gene = rng.choice(gene_list, size=n_test, replace=False).tolist()
    print(test_gene)
    train_gene = [g for g in gene_list if g not in test_gene]
    print(train_gene)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_gene是non-targeting
    adata_train = adata[adata.obs[condition_keys].isin(train_gene)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_gene)].copy()
    del adata

['CHMP3', 'AKT2', 'SHPRH', 'TMSB4X', 'KLF10', 'TARBP2', 'KDM2B', 'SV2A', 'CLDN6', 'TCF3', 'ANTXR1', 'NDUFB6', 'TADA1', 'MED12', 'CAMSAP2', 'IDE', 'PRCP', 'WFS1', 'FOXH1', 'SMARCA4', 'TWF2', 'SAFB', 'POLB', 'TSC22D4', 'ACVR1B', 'PMS1', 'NISCH', 'INSIG1', 'DHCR24', 'MAP3K7', 'TMSB10', 'SMARCA5', 'STAG2', 'ZNF426', 'DNMT1', 'SSBP1', 'HIRA', 'USP22', 'PBX1', 'EID2', 'KAT2A', 'MAPKAPK2', 'SRC', 'HSBP1', 'MED13', 'ZNF593', 'TET1', 'KDR', 'EIF4B', 'SIX4', 'TFAM', 'MAST2', 'SMAGP', 'CAST', 'MTA1', 'ATP1B1', 'KIF20A', 'KIF1B', 'NCK2', 'XRCC4', 'RNF2', 'CASP3', 'HMGN1', 'GSK3B', 'RAB3B', 'MED13L', 'HMGB2', 'DHX36', 'IGF2R', 'HAT1', 'STAT6', 'ARID1A', 'CDCA2', 'SNCA', 'TRAM2', 'STAT1', 'TMBIM6', 'PMEL', 'DAXX', 'PRDM14', 'NREP', 'CREG1', 'METTL14', 'RAF1', 'RRM1', 'UQCRQ', 'BIRC2', 'IKBKG', 'LRPPRC', 'SDC1', 'ZNF714', 'CASP2', 'STX4', 'IRF3', 'CENPO', 'EWSR1', 'PLCB3', 'PHF14', 'MAU2', 'TAZ', 'DLG5', 'CLDN7', 'PAGR1', 'PTPN1', 'NDUFB4', 'KDM1A', 'UBE3C', 'TGFBR2', 'TRAPPC6A', 'MED24', 'METTL17', 

In [10]:
n_comps = 128
n_hidden = 2048
n_layers = 2
condition_rep_dict = pd.read_pickle("./data/processed/vcc_data_target_genes_embedding.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1]
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

INFO     No backup URL provided for missing file                                                                   
         ./data/processed/model/X_flatvi_ncomps128_hidden2048_layers2_ArcVirtualCell_ref/model.pt                  


In [11]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    condition_rep_dict = condition_rep_dict,
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
if sample_rep == "X_pca":
    sample_rep_scaled = sample_rep + "_scaled" # 额 别忘了
else:
    sample_rep_scaled = sample_rep

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/pytorch_lightning/core/optimizer.py:259: Found unsupported keys in the lr scheduler dict: {'min_lr', 'threshold'}. HINT: remove them from the output of `configure_optimizers`.


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                 ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_layers       │ MLP    │ 41.2 M │ train │     0 │
│ 1 │ decoder_layers       │ MLP    │  4.5 M │ train │     0 │
│ 2 │ library_size_decoder │ Linear │    129 │ train │     0 │
│ 3 │ decoder_mu_lib       │ Linear │ 37.0 M │ train │     0 │
│ 4 │ mu_logvar            │ Linear │  524 K │ train │     0 │
│   │ other params         │ n/a    │ 18.1 K │ n/a   │   n/a │
└───┴──────────────────────┴────────┴────────┴───────┴───────┘

Trainable params: 83.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 83.3 M                                                                                               
Total estimated model params size (MB): 333                                                                        
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

Model saved to ./data/processed/model/X_flatvi_ncomps128_hidden2048_layers2_ArcVirtualCell/model.pt
Control Std: [1.4382958 5.3162417 3.7127042 3.4208639 7.4486   ]
Train Std: [1.4360173 5.3206215 3.7339466 3.4221213 7.4640083]
Test Std: [1.4274989 5.303356  3.7206888 3.4085817 7.4403133]


In [12]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}_{n_comps}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [13]:
print(adata_control)
print(adata_train)
print(adata_test)

AnnData object with n_obs × n_vars = 38176 × 18080
    obs: 'target_gene', 'guide_id', 'batch', 'is_control'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt', 'ribo', 'n_cells'
    obsm: 'X_flatvi'
    layers: 'counts'
AnnData object with n_obs × n_vars = 164787 × 18080
    obs: 'target_gene', 'guide_id', 'batch', 'is_control'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt', 'ribo', 'n_cells'
    obsm: 'X_flatvi', 'gene_embeddings'
    layers: 'counts'
AnnData object with n_obs × n_vars = 18310 × 18080
    obs: 'target_gene', 'guide_id', 'batch', 'is_control'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt', 'ribo', 'n_cells'
    obsm: 'X_flatvi', 'gene_embeddings'
    layers: 'counts'


In [14]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

AttributeError: 'FlatVIEmbedding' object has no attribute 'get_normalized_expression'

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()